<a href="https://colab.research.google.com/github/sangramshitole07/AML/blob/main/Feature_Select_Wrapper_Methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from sklearn.datasets import load_breast_cancer

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.pipeline import Pipeline

#1)Forward feature Selections

In [5]:
data=load_breast_cancer()
X=data.data
y=data.target

In [4]:
print(X.shape)

(569, 30)


In [6]:
model=Pipeline([
    ('scaler',StandardScaler()),
    ('classifier',LogisticRegression(max_iter=5000))
])

selector=SequentialFeatureSelector(
    model,
    n_features_to_select=5,
    direction='forward',
    scoring='accuracy',
    cv=5
)

selector.fit(X,y)
print(data.feature_names[selector.get_support()])

['mean symmetry' 'concavity error' 'worst texture' 'worst perimeter'
 'worst smoothness']


#2)Backward feature elimination

In [7]:
selector=SequentialFeatureSelector(
    model,
    n_features_to_select=5,
    direction='backward',
    scoring='accuracy',
    cv=5
)

selector.fit(X,y)
print(data.feature_names[selector.get_support()])


['mean texture' 'mean concave points' 'worst area' 'worst smoothness'
 'worst concavity']


#3)Recursive Feature Selections

In [9]:
from sklearn.feature_selection import RFE
data=load_breast_cancer()
X=data.data
y=data.target

X_scaled=StandardScaler().fit_transform(X)
model=LogisticRegression(max_iter=5000)
selector=RFE(
    estimator=model,
    n_features_to_select=5,
    step=1#remove feature 1 at a time
)
selector.fit(X_scaled,y)

for name,selected,rank in zip(
    data.feature_names,
    selector.get_support(),
    selector.ranking_
):
    print(
        f"{name}:{selected}:{rank}"
)

mean radius:False:12
mean texture:False:14
mean perimeter:False:15
mean area:False:8
mean smoothness:False:23
mean compactness:False:10
mean concavity:False:13
mean concave points:False:3
mean symmetry:False:25
mean fractal dimension:False:20
radius error:True:1
texture error:False:22
perimeter error:False:9
area error:False:4
smoothness error:False:18
compactness error:False:6
concavity error:False:24
concave points error:False:21
symmetry error:False:17
fractal dimension error:False:16
worst radius:True:1
worst texture:True:1
worst perimeter:False:2
worst area:True:1
worst smoothness:False:7
worst compactness:False:26
worst concavity:False:5
worst concave points:True:1
worst symmetry:False:11
worst fractal dimension:False:19


using cross validation

In [13]:
#use stratify when we have imbalanced data and not cross validation
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

model=LogisticRegression(max_iter=5000)
cv=StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

selector=RFECV(
    estimator=model,
    step=1,
    cv=cv,
    scoring='accuracy',
    min_features_to_select=1
)

selector.fit(X_scaled,y)

print("optimal number of features : ", selector.n_features_)
print("selected features : ", data.feature_names[selector.support_])

for name,selected,rank in zip(
    data.feature_names,
    selector.get_support(),
    selector.ranking_
):
    print(f"{name}:{selected}:{rank}")

optimal number of features :  23
selected features :  ['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean compactness' 'mean concavity' 'mean concave points' 'radius error'
 'perimeter error' 'area error' 'smoothness error' 'compactness error'
 'symmetry error' 'fractal dimension error' 'worst radius' 'worst texture'
 'worst perimeter' 'worst area' 'worst smoothness' 'worst concavity'
 'worst concave points' 'worst symmetry' 'worst fractal dimension']
mean radius:True:1
mean texture:True:1
mean perimeter:True:1
mean area:True:1
mean smoothness:False:5
mean compactness:True:1
mean concavity:True:1
mean concave points:True:1
mean symmetry:False:7
mean fractal dimension:False:2
radius error:True:1
texture error:False:4
perimeter error:True:1
area error:True:1
smoothness error:True:1
compactness error:True:1
concavity error:False:6
concave points error:False:3
symmetry error:True:1
fractal dimension error:True:1
worst radius:True:1
worst texture:True:1
worst perimeter:True:1


#4)Embedded Method


In [16]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
data=load_diabetes()
X=pd.DataFrame(data.data, columns=data.feature_names)
y=data.target

scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

lasso=LassoCV(cv=5,random_state=42, max_iter=10000)

lasso.fit(X_scaled,y)
coefficients =pd.Series(
    lasso.coef_,
    index=X.columns
)
selected_features =coefficients[
    coefficients.abs() > 1e-6

]

print("Best alpha :",lasso.alpha_)
print("Selected Features :", selected_features)

Best alpha : 0.07891843500595844
Selected Features : age    -0.308921
sex   -11.225046
bmi    24.816849
bp     15.271304
s1    -27.085410
s2     14.386231
s4      6.835041
s5     31.864972
s6      3.179041
dtype: float64


#L1 regularized logistic Regression



In [19]:
from sklearn.feature_selection import SelectFromModel
#means we have to use machine learning model

data=load_breast_cancer()
X=data.data
y=data.target

pipeline=Pipeline([
    ("scaler",StandardScaler()),
    ("selection",SelectFromModel(LogisticRegression(max_iter=5000,penalty="l1",solver="liblinear",C=0.1))),
    ("classifier",LogisticRegression(max_iter=5000))
])

pipeline.fit(X,y)
mask=pipeline.named_steps["selection"].get_support()
print(data.feature_names[mask])


['mean concave points' 'radius error' 'worst radius' 'worst texture'
 'worst smoothness' 'worst concavity' 'worst concave points'
 'worst symmetry']


In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.feature_selection import SelectFromModel
iris=load_iris()
clf=RandomForestClassifier(n_estimators=10)
clf=clf.fit(X,y)
clf.feature_importances_
#TODO :select only two most important feature

array([0.00619662, 0.01624825, 0.06548824, 0.00704306, 0.        ,
       0.00429324, 0.06492757, 0.28608106, 0.00485579, 0.00147638,
       0.0032388 , 0.0058236 , 0.00588937, 0.01528478, 0.00170719,
       0.00500171, 0.01466927, 0.00418105, 0.0040364 , 0.0124408 ,
       0.23486654, 0.01140294, 0.02387691, 0.04360305, 0.01883184,
       0.03931361, 0.02252029, 0.06686853, 0.00765491, 0.00217821])

In [23]:
import numpy as np
# Select the top 2 features using max_features
model_selector = SelectFromModel(clf, prefit=True, threshold=-np.inf, max_features=2)
X_new = model_selector.transform(X)
print(X_new.shape)

(569, 2)


In [24]:
#feature selection using random forest

In [25]:
#feature selection using pipleline and kneighborClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
clfs=Pipeline([('feature_selection',
                 SelectFromModel(RandomForestClassifier(n_estimators=10))),
                ('classification ,'KNeighborsClassifier)])
model=clfs.fit(X,y)
model.score(X,y)

In [26]:
#using stratify

In [27]:
#take your own dataset and apply this
#find the prediction accuracy , confusion , classification report ,roc curve